# RQ3.2 — Class-level Analysis

This notebook reproduces the descriptive class-level comparison between the **zero-shot baseline** (`p0_baseline`) and the final **code-disambiguation** prompt (`p_code_disambiguation`).

The analysis is intentionally separate from the project-level repeated-measures analyses used in the four RQ3.2 prompt-refinement stages.

Here, precision, recall, and F1 are computed from predictions **pooled across all commits**. These results are diagnostic and are therefore not directly comparable to the mean project-level Macro-F1 values used in the inferential analyses.

Expected replication-package layout:

```text
results/rq3_2/
├── analysis/
│   └── rq3_2_class_analysis.ipynb
├── metrics/
│   └── class_metrics_baseline_vs_code_disambiguation.csv
├── predictions/
│   └── predictions.csv.gz
└── figures/
    └── rq3_class_f1.png
```

The notebook recomputes the class-level metrics directly from `predictions.csv.gz`. If the metrics CSV already exists, it validates the recomputed values against it. Otherwise, it creates the CSV.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import precision_recall_fscore_support

EXPECTED_COMMITS = 7937
BASELINE_PROMPT = "p0_baseline"
FINAL_PROMPT = "p_code_disambiguation"

CLASSES = [
    "fix",
    "feat",
    "build",
    "ci",
    "docs",
    "style",
    "refactor",
    "perf",
    "test",
    "chore",
    "others",
]

PROMPT_LABELS = {
    BASELINE_PROMPT: "Baseline",
    FINAL_PROMPT: "Code disambiguation",
}

# Optional override. Set this to results/rq3_2 if automatic path resolution
# does not find the package files.
RQ3_2_ROOT_OVERRIDE = None


def candidate_roots():
    roots = []

    if RQ3_2_ROOT_OVERRIDE is not None:
        roots.append(Path(RQ3_2_ROOT_OVERRIDE).expanduser().resolve())

    cwd = Path.cwd().resolve()

    # Common launch locations:
    #   results/rq3_2/analysis/
    #   results/rq3_2/
    #   results/
    #   repository root
    roots.append(cwd)
    roots.extend(list(cwd.parents)[:5])

    unique = []
    seen = set()
    for root in roots:
        marker = str(root)
        if marker not in seen:
            unique.append(root)
            seen.add(marker)
    return unique


def resolve_rq3_2_root():
    candidates = []

    for root in candidate_roots():
        candidates.extend([
            root,
            root / "rq3_2",
            root / "results" / "rq3_2",
        ])

    unique = []
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        marker = str(candidate)
        if marker in seen:
            continue
        seen.add(marker)
        unique.append(candidate)

        if (
            (candidate / "predictions" / "predictions.csv.gz").is_file()
            and (candidate / "metrics").is_dir()
        ):
            return candidate

    attempted = "\n".join(f"  - {path}" for path in unique)
    raise FileNotFoundError(
        "Could not locate results/rq3_2.\n"
        f"Checked:\n{attempted}\n\n"
        "Set RQ3_2_ROOT_OVERRIDE explicitly if your package uses a "
        "different location."
    )


RQ3_2_ROOT = resolve_rq3_2_root()
PREDICTIONS_PATH = RQ3_2_ROOT / "predictions" / "predictions.csv.gz"
METRICS_PATH = (
    RQ3_2_ROOT
    / "metrics"
    / "class_metrics_baseline_vs_code_disambiguation.csv"
)
FIGURES_DIR = RQ3_2_ROOT / "figures"
FIGURE_PATH = FIGURES_DIR / "rq3_class_f1.png"

print(f"RQ3.2 root : {RQ3_2_ROOT}")
print(f"Predictions: {PREDICTIONS_PATH}")
print(f"Metrics    : {METRICS_PATH}")
print(f"Figure     : {FIGURE_PATH}")


## 1. Load and validate the final predictions

Only the two configurations used in the class-level comparison are retained. The notebook verifies that both contain the full 7,937-commit cohort and that their targets refer to the same commits.


In [ ]:
predictions = pd.read_csv(
    PREDICTIONS_PATH,
    compression="gzip",
    dtype={
        "prompt": "string",
        "project": "string",
        "commit_hash": "string",
        "target": "string",
        "prediction": "string",
    },
)

required_columns = {
    "prompt",
    "project",
    "commit_hash",
    "target",
    "prediction",
}
missing = sorted(required_columns - set(predictions.columns))
if missing:
    raise ValueError(
        f"Missing required columns in predictions.csv.gz: {missing}"
    )

comparison = predictions.loc[
    predictions["prompt"].isin([BASELINE_PROMPT, FINAL_PROMPT]),
    ["prompt", "project", "commit_hash", "target", "prediction"],
].copy()

if len(comparison) != EXPECTED_COMMITS * 2:
    raise ValueError(
        f"Expected {EXPECTED_COMMITS * 2:,} rows for the two prompts, "
        f"found {len(comparison):,}."
    )

counts = (
    comparison.groupby("prompt")
    .agg(
        rows=("commit_hash", "size"),
        projects=("project", "nunique"),
        commits=("commit_hash", "nunique"),
    )
)

display(counts)

for prompt in [BASELINE_PROMPT, FINAL_PROMPT]:
    if counts.loc[prompt, "rows"] != EXPECTED_COMMITS:
        raise ValueError(
            f"{prompt}: expected {EXPECTED_COMMITS:,} rows, "
            f"found {counts.loc[prompt, 'rows']:,}."
        )

duplicate_keys = comparison.duplicated(
    ["prompt", "project", "commit_hash"],
    keep=False,
)
if duplicate_keys.any():
    raise ValueError(
        "Duplicate prompt/project/commit_hash rows were found."
    )

baseline_keys = (
    comparison.loc[
        comparison["prompt"] == BASELINE_PROMPT,
        ["project", "commit_hash", "target"],
    ]
    .sort_values(["project", "commit_hash"])
    .reset_index(drop=True)
)

final_keys = (
    comparison.loc[
        comparison["prompt"] == FINAL_PROMPT,
        ["project", "commit_hash", "target"],
    ]
    .sort_values(["project", "commit_hash"])
    .reset_index(drop=True)
)

if not baseline_keys.equals(final_keys):
    raise ValueError(
        "Baseline and code-disambiguation do not contain the same "
        "project/commit/target cohort."
    )

unknown_targets = sorted(
    set(comparison["target"].dropna()) - set(CLASSES)
)
if unknown_targets:
    raise ValueError(
        f"Unexpected ground-truth classes: {unknown_targets}"
    )

print("Cohort validation: OK")


## 2. Recompute pooled class-level precision, recall, F1, and support

Predictions outside the 11-class taxonomy, such as strict invalid outputs or final abstentions, remain incorrect predictions. They are **not mapped to `others`** and are **not introduced as semantic classes** in the metric table.

By explicitly passing the 11 evaluated classes to `precision_recall_fscore_support`, such outputs still contribute false negatives to their ground-truth classes while remaining outside the semantic label set.


In [ ]:
def compute_class_metrics(
    frame: pd.DataFrame,
    prompt: str,
) -> pd.DataFrame:
    subset = frame.loc[frame["prompt"] == prompt].copy()

    precision, recall, f1, support = precision_recall_fscore_support(
        subset["target"],
        subset["prediction"],
        labels=CLASSES,
        average=None,
        zero_division=0,
    )

    return pd.DataFrame({
        "policy": "strict_taxonomy",
        "prompt": prompt,
        "prompt_label": PROMPT_LABELS[prompt],
        "class": CLASSES,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support.astype(int),
    })


baseline_metrics = compute_class_metrics(
    comparison,
    BASELINE_PROMPT,
)
final_metrics = compute_class_metrics(
    comparison,
    FINAL_PROMPT,
)

if not np.array_equal(
    baseline_metrics["support"].to_numpy(),
    final_metrics["support"].to_numpy(),
):
    raise RuntimeError(
        "Class support differs between the two prompt configurations."
    )

delta_metrics = pd.DataFrame({
    "policy": "strict_taxonomy",
    "prompt": "delta_code_disambiguation_minus_baseline",
    "prompt_label": "Code disambiguation − Baseline",
    "class": CLASSES,
    "precision": (
        final_metrics["precision"].to_numpy()
        - baseline_metrics["precision"].to_numpy()
    ),
    "recall": (
        final_metrics["recall"].to_numpy()
        - baseline_metrics["recall"].to_numpy()
    ),
    "f1": (
        final_metrics["f1"].to_numpy()
        - baseline_metrics["f1"].to_numpy()
    ),
    "support": baseline_metrics["support"].to_numpy(),
})

computed_metrics = pd.concat(
    [baseline_metrics, final_metrics, delta_metrics],
    ignore_index=True,
)

display(
    computed_metrics.style.format({
        "precision": "{:.6f}",
        "recall": "{:.6f}",
        "f1": "{:.6f}",
    })
)


## 3. Validate or create the class-level metrics CSV

If `class_metrics_baseline_vs_code_disambiguation.csv` already exists, the notebook verifies its schema, row keys, supports, and metric values against the predictions. This prevents a stale CSV from silently diverging from the published prediction artifact.

If it does not exist, the recomputed table is written to `results/rq3_2/metrics/`.


In [ ]:
KEY_COLUMNS = [
    "policy",
    "prompt",
    "prompt_label",
    "class",
]
VALUE_COLUMNS = [
    "precision",
    "recall",
    "f1",
    "support",
]

if METRICS_PATH.is_file():
    published_metrics = pd.read_csv(METRICS_PATH)

    expected_columns = KEY_COLUMNS + VALUE_COLUMNS
    if list(published_metrics.columns) != expected_columns:
        raise ValueError(
            "Unexpected metrics CSV schema.\n"
            f"Expected: {expected_columns}\n"
            f"Found:    {list(published_metrics.columns)}"
        )

    expected_sorted = (
        computed_metrics
        .sort_values(KEY_COLUMNS)
        .reset_index(drop=True)
    )
    published_sorted = (
        published_metrics
        .sort_values(KEY_COLUMNS)
        .reset_index(drop=True)
    )

    if published_sorted[KEY_COLUMNS].astype(str).equals(
        expected_sorted[KEY_COLUMNS].astype(str)
    ) is False:
        raise ValueError(
            "The metrics CSV contains different row keys from the "
            "metrics recomputed from predictions."
        )

    if not np.array_equal(
        published_sorted["support"].to_numpy(dtype=int),
        expected_sorted["support"].to_numpy(dtype=int),
    ):
        raise ValueError(
            "Class support in the metrics CSV does not match the predictions."
        )

    for metric in ["precision", "recall", "f1"]:
        if not np.allclose(
            published_sorted[metric].to_numpy(dtype=float),
            expected_sorted[metric].to_numpy(dtype=float),
            rtol=0,
            atol=1e-12,
            equal_nan=True,
        ):
            max_difference = np.max(
                np.abs(
                    published_sorted[metric].to_numpy(dtype=float)
                    - expected_sorted[metric].to_numpy(dtype=float)
                )
            )
            raise ValueError(
                f"{metric}: published CSV differs from recomputed values. "
                f"Maximum absolute difference = {max_difference:.3e}"
            )

    print("Published class-metrics CSV matches predictions exactly.")

else:
    METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    computed_metrics.to_csv(
        METRICS_PATH,
        index=False,
        float_format="%.15g",
    )
    print(f"Created: {METRICS_PATH}")


## 4. Diagnostic comparison used in the paper

The table below places baseline F1, final-prompt F1, \(\Delta F_1\), and support side by side. Classes are ordered by the F1 change produced by code disambiguation.


In [ ]:
class_comparison = (
    baseline_metrics[
        ["class", "precision", "recall", "f1", "support"]
    ]
    .rename(columns={
        "precision": "baseline_precision",
        "recall": "baseline_recall",
        "f1": "baseline_f1",
    })
    .merge(
        final_metrics[
            ["class", "precision", "recall", "f1"]
        ].rename(columns={
            "precision": "code_precision",
            "recall": "code_recall",
            "f1": "code_f1",
        }),
        on="class",
        validate="one_to_one",
    )
)

class_comparison["delta_precision"] = (
    class_comparison["code_precision"]
    - class_comparison["baseline_precision"]
)
class_comparison["delta_recall"] = (
    class_comparison["code_recall"]
    - class_comparison["baseline_recall"]
)
class_comparison["delta_f1"] = (
    class_comparison["code_f1"]
    - class_comparison["baseline_f1"]
)

class_comparison = class_comparison.sort_values(
    "delta_f1",
    ascending=False,
).reset_index(drop=True)

display(
    class_comparison[
        [
            "class",
            "support",
            "baseline_f1",
            "code_f1",
            "delta_f1",
            "baseline_precision",
            "code_precision",
            "delta_precision",
            "baseline_recall",
            "code_recall",
            "delta_recall",
        ]
    ].style.format({
        "baseline_f1": "{:.3f}",
        "code_f1": "{:.3f}",
        "delta_f1": "{:+.3f}",
        "baseline_precision": "{:.3f}",
        "code_precision": "{:.3f}",
        "delta_precision": "{:+.3f}",
        "baseline_recall": "{:.3f}",
        "code_recall": "{:.3f}",
        "delta_recall": "{:+.3f}",
    })
)


## 5. Reproduce the class-level F1 figure

The figure mirrors the paper's class-level comparison: circles represent the baseline, diamonds represent code disambiguation, and classes are ordered by \(\Delta F_1\). Class support is shown in parentheses.

The PNG is written to `results/rq3_2/figures/rq3_class_f1.png`.


In [ ]:
plot_data = class_comparison.copy()

y = np.arange(len(plot_data))
baseline_f1 = plot_data["baseline_f1"].to_numpy()
code_f1 = plot_data["code_f1"].to_numpy()

fig, ax = plt.subplots(figsize=(8, 5.8))

for index, (baseline_value, code_value) in enumerate(
    zip(baseline_f1, code_f1)
):
    ax.plot(
        [baseline_value, code_value],
        [index, index],
        linewidth=1,
        alpha=0.55,
    )

ax.scatter(
    baseline_f1,
    y,
    marker="o",
    label="Baseline",
    zorder=3,
)

ax.scatter(
    code_f1,
    y,
    marker="D",
    label="Code disambiguation",
    zorder=3,
)

labels = [
    f"{row['class']} ({int(row['support'])})"
    for _, row in plot_data.iterrows()
]

ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("F1-score")
ax.set_ylabel("Class (support)")
ax.set_xlim(left=0)
ax.grid(axis="x", alpha=0.25)
ax.legend()
fig.tight_layout()

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(
    FIGURE_PATH,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"Saved figure: {FIGURE_PATH}")


## 6. Data-driven summary

This cell reports the main descriptive changes without hard-coding the values used in the article.


In [ ]:
largest_gain = class_comparison.loc[
    class_comparison["delta_f1"].idxmax()
]
largest_decline = class_comparison.loc[
    class_comparison["delta_f1"].idxmin()
]

print(
    "Largest F1 improvement:"
    f" {largest_gain['class']} "
    f"{largest_gain['baseline_f1']:.3f} -> "
    f"{largest_gain['code_f1']:.3f} "
    f"(delta={largest_gain['delta_f1']:+.3f})"
)

print(
    "Largest F1 decline:"
    f" {largest_decline['class']} "
    f"{largest_decline['baseline_f1']:.3f} -> "
    f"{largest_decline['code_f1']:.3f} "
    f"(delta={largest_decline['delta_f1']:+.3f})"
)

print("\nOrdered F1 changes:")
for _, row in class_comparison.iterrows():
    print(
        f"  {row['class']:<8} "
        f"{row['baseline_f1']:.3f} -> "
        f"{row['code_f1']:.3f} "
        f"({row['delta_f1']:+.3f}), "
        f"support={int(row['support'])}"
    )


## Reproducibility notes

- This is a **descriptive pooled class-level analysis**, not a second inferential analysis.
- The project-level Macro-F1 analyses remain in `rq3_2_prompt_refinement.ipynb`.
- `others` is treated as a valid semantic class only when explicitly predicted.
- Strict invalid outputs and final abstentions remain incorrect predictions but are not introduced as semantic classes.
- The class-metrics CSV is derived from `predictions.csv.gz` and can be validated or regenerated by this notebook.
- The figure is generated from the same recomputed metrics, keeping the table, text, and visualization tied to the published prediction artifact.
